In [3]:
import pandas as pd
import numpy as np
import joblib
from scipy.stats import wilcoxon, spearmanr
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import NearMiss
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("Ablation Study: Selection Strategy Comparison (Same CF Pool)")
print("  Condition A: No immutable constraint  + Random selection")
print("  Condition B: Immutable constraint     + Random selection (first valid)")
print("  Condition C: Immutable constraint     + Quality Score selection [Proposed]")
print("=" * 80)

# ============================================================================
# 1. Load Data
# ============================================================================
cf_all   = pd.read_csv('cf_results_full.csv')       # 1,064 rows (266 firms × 4 CFs)
model     = joblib.load('base_model_final.pkl')
features  = joblib.load('selected_features_final.pkl')
threshold = joblib.load('base_model_threshold_final.pkl')

# ── Step2와 완전히 동일한 절차로 scaler / IsolationForest 재현 ──────────────
df_orig = pd.read_csv('selected_data_for_modeling.csv')
X_full  = df_orig.drop(['ID', 'PERF_12M'], axis=1)[features]
y_full  = df_orig['PERF_12M']

# 1) NearMiss (전체 데이터 기준, Step2와 동일)
nm = NearMiss(version=1, n_neighbors=3)
X_resampled, y_resampled = nm.fit_resample(X_full, y_full)

# 2) Train/Test split (80:20, stratify, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled,
    test_size=0.2, random_state=42, stratify=y_resampled
)

# 3) MinMaxScaler — X_train 기준으로 fit
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)

# 4) IsolationForest — X_train_scaled 기준으로 fit
iso = IsolationForest(n_estimators=100, contamination='auto',
                      random_state=42, n_jobs=-1)
iso.fit(X_train_scaled)

print(f"Scaler & IsolationForest fitted on X_train (n={len(X_train)}).")

# Realism 계산용: solvent firms (원본 전체 데이터 기준)
solvent_X = df_orig[df_orig['PERF_12M'] == 0][features]

print(f"Total CF candidates : {len(cf_all)} rows")
print(f"Unique firms        : {cf_all['ID'].nunique()}")
print(f"Threshold           : {threshold:.4f}")

# ============================================================================
# 2. Define Immutable Features
# ============================================================================
IMMUTABLE = [
    'FN1_11_1', 'FN1_13_1', 'FN2_1_1', 'FN2_3_2', 'FN2_5_1',
    'FN2_10_1', 'FN3_11_1', 'FN1_24_1'
]

# ============================================================================
# 3. Metric Computation Functions
# ============================================================================

def get_cf_feature_array(row):
    return np.array([row[f'CF_{f}'] for f in features])

def get_orig_feature_array(row):
    return np.array([row[f'Original_{f}'] for f in features])

def compute_validity(row, model, threshold):
    cf_arr = get_cf_feature_array(row).reshape(1, -1)
    prob   = model.predict_proba(cf_arr)[0, 1]
    return 1.0 if prob < threshold else 0.0

def compute_proximity(row, scaler):
    """Step4와 동일: StandardScaler 정규화 공간에서 L1 거리."""
    orig = get_orig_feature_array(row)
    cf   = get_cf_feature_array(row)
    orig_scaled = scaler.transform(orig.reshape(1, -1))[0]
    cf_scaled   = scaler.transform(cf.reshape(1, -1))[0]
    return float(np.mean(np.abs(orig_scaled - cf_scaled)))

def compute_sparsity(row, scaler, tol=1e-6):
    """Step4와 동일: 정규화 공간에서 변화량이 tol 미만인 feature 비율."""
    orig = get_orig_feature_array(row)
    cf   = get_cf_feature_array(row)
    orig_scaled = scaler.transform(orig.reshape(1, -1))[0]
    cf_scaled   = scaler.transform(cf.reshape(1, -1))[0]
    changes  = np.abs(orig_scaled - cf_scaled)
    return float(np.mean(changes < tol))

def compute_robustness(row, model, threshold, n=10, noise_level=0.01):
    """Step4와 동일: 원본 스케일에서 비율 노이즈 추가."""
    cf_arr = get_cf_feature_array(row)
    count  = 0
    for _ in range(n):
        noise     = np.random.normal(0, noise_level, size=cf_arr.shape)
        perturbed = cf_arr + (noise * (cf_arr + 1e-6))
        if model.predict_proba(perturbed.reshape(1, -1))[0, 1] < threshold:
            count += 1
    return count / n

def check_immutable_violated(row):
    """Returns True if any immutable feature was changed (violation)."""
    for f in IMMUTABLE:
        if f in features:
            if abs(row[f'Original_{f}'] - row[f'CF_{f}']) > 1e-6:
                return True
    return False

def compute_quality_score(v, prox, sp, real, rob):
    return (5 * v + (1 - prox) + sp + real + rob) / 9

# ============================================================================
# 4. Pre-compute Metrics for All 1,064 CF Candidates
# ============================================================================
print("\n[Step 1] Computing metrics for all CF candidates...")

# Fit Isolation Forest on solvent firms
iso = IsolationForest(contamination=0.1, random_state=42)
iso.fit(solvent_X)

records = []
for _, row in cf_all.iterrows():
    v    = compute_validity(row, model, threshold)
    prox = compute_proximity(row, scaler)
    sp   = compute_sparsity(row, scaler)
    cf_arr = get_cf_feature_array(row).reshape(1, -1)
    cf_scaled = scaler.transform(cf_arr)
    iso_score = iso.decision_function(cf_scaled)[0]
    real = float((iso_score + 0.5)) if iso_score > -0.5 else 0.0
    rob  = compute_robustness(row, model, threshold)
    qs   = compute_quality_score(v, prox, sp, real, rob)
    immut_violated = check_immutable_violated(row)

    records.append({
        'ID': row['ID'],
        'CF_Number': row['CF_Number'],
        'Validity': v,
        'Proximity': prox,
        'Sparsity': sp,
        'Realism': real,
        'Robustness': rob,
        'QualityScore': qs,
        'Immutable_Violated': immut_violated,
    })

metrics_df = pd.DataFrame(records)
print(f"Done. Total rows: {len(metrics_df)}")

# ============================================================================
# 5. Select Best CF per Firm under Each Condition
# ============================================================================
print("\n[Step 2] Selecting best CF per firm under each condition...")

ids = cf_all['ID'].unique()
results = {'A': [], 'B': [], 'C': []}

np.random.seed(42)

for firm_id in ids:
    firm = metrics_df[metrics_df['ID'] == firm_id].copy()

    # --- Condition A: No immutable constraint, random selection ---
    pool_A = firm  # all 4 CFs
    sel_A  = pool_A.sample(1, random_state=42).iloc[0]
    results['A'].append(sel_A)

    # --- Condition B: Immutable constraint, random selection among valid ---
    pool_B = firm[~firm['Immutable_Violated']]
    if pool_B.empty:
        pool_B = firm  # fallback (shouldn't happen with genetic method)
    sel_B = pool_B.sample(1, random_state=42).iloc[0]
    results['B'].append(sel_B)

    # --- Condition C: Immutable constraint + Quality Score (Proposed) ---
    pool_C = firm[~firm['Immutable_Violated']]
    if pool_C.empty:
        pool_C = firm
    sel_C = pool_C.loc[pool_C['QualityScore'].idxmax()]
    results['C'].append(sel_C)

df_A = pd.DataFrame(results['A'])
df_B = pd.DataFrame(results['B'])
df_C = pd.DataFrame(results['C'])

# ============================================================================
# 6. Summarise Results
# ============================================================================
METRICS = ['Validity', 'Proximity', 'Sparsity', 'Realism', 'Robustness', 'QualityScore']

print("\n" + "=" * 80)
print("ABLATION STUDY RESULTS (n=266 firms each condition)")
print("=" * 80)

summary = {}
for cond, df in [('A (No constraint + Random)', df_A),
                 ('B (Immutable + Random)',     df_B),
                 ('C (Immutable + QScore) [Proposed]', df_C)]:
    row = {}
    for m in METRICS:
        row[m] = f"{df[m].mean():.4f} ± {df[m].std():.4f}"
    summary[cond] = row

summary_df = pd.DataFrame(summary).T
print(summary_df.to_string())

# ============================================================================
# 7. Statistical Tests: A vs C and B vs C (Wilcoxon signed-rank)
# ============================================================================
print("\n[Step 3] Statistical Tests (Wilcoxon signed-rank, A vs C and B vs C)")
print("-" * 60)

for m in ['Proximity', 'Sparsity', 'Realism', 'QualityScore']:
    stat_ac, p_ac = wilcoxon(df_A[m], df_C[m])
    stat_bc, p_bc = wilcoxon(df_B[m], df_C[m])

    # Effect size: rank-biserial correlation
    n = len(df_A)
    r_ac = 1 - (2 * stat_ac) / (n * (n + 1))
    r_bc = 1 - (2 * stat_bc) / (n * (n + 1))

    print(f"\n{m}:")
    print(f"  A vs C → p={p_ac:.4f}, effect r={r_ac:.3f} {'***' if p_ac<0.001 else '**' if p_ac<0.01 else '*' if p_ac<0.05 else 'ns'}")
    print(f"  B vs C → p={p_bc:.4f}, effect r={r_bc:.3f} {'***' if p_bc<0.001 else '**' if p_bc<0.01 else '*' if p_bc<0.05 else 'ns'}")

# ============================================================================
# 8. Sensitivity Analysis: Quality Score Weight Schemes
# ============================================================================
print("\n[Step 4] Sensitivity Analysis: Weight Scheme Comparison")
print("-" * 60)

weight_schemes = {
    'Proposed (5,1,1,1,1)': (5, 1, 1, 1, 1),
    'Equal (1,1,1,1,1)':    (1, 1, 1, 1, 1),
    'Prox-heavy (5,3,1,1,1)': (5, 3, 1, 1, 1),
    'Real-heavy (5,1,1,3,1)': (5, 1, 1, 3, 1),
    'Rob-heavy (5,1,1,1,3)':  (5, 1, 1, 1, 3),
}

# For each firm, compute rank of CFs under each weight scheme (using pool C)
rank_corr_results = {}
firm_ids_list = list(ids)

proposed_ranks = []
scheme_ranks   = {s: [] for s in weight_schemes if s != 'Proposed (5,1,1,1,1)'}

for firm_id in firm_ids_list:
    firm = metrics_df[(metrics_df['ID'] == firm_id) & (~metrics_df['Immutable_Violated'])].copy()
    if firm.empty:
        firm = metrics_df[metrics_df['ID'] == firm_id].copy()

    denom_proposed = sum(weight_schemes['Proposed (5,1,1,1,1)'])
    firm['QS_proposed'] = (
        weight_schemes['Proposed (5,1,1,1,1)'][0] * firm['Validity'] +
        weight_schemes['Proposed (5,1,1,1,1)'][1] * (1 - firm['Proximity']) +
        weight_schemes['Proposed (5,1,1,1,1)'][2] * firm['Sparsity'] +
        weight_schemes['Proposed (5,1,1,1,1)'][3] * firm['Realism'] +
        weight_schemes['Proposed (5,1,1,1,1)'][4] * firm['Robustness']
    ) / denom_proposed

    best_proposed_cfnum = firm.loc[firm['QS_proposed'].idxmax(), 'CF_Number']

    for scheme_name, (wv, wp, ws, wr, wrob) in weight_schemes.items():
        if scheme_name == 'Proposed (5,1,1,1,1)':
            continue
        denom = wv + wp + ws + wr + wrob
        firm[f'QS_{scheme_name}'] = (
            wv * firm['Validity'] +
            wp * (1 - firm['Proximity']) +
            ws * firm['Sparsity'] +
            wr * firm['Realism'] +
            wrob * firm['Robustness']
        ) / denom
        best_other_cfnum = firm.loc[firm[f'QS_{scheme_name}'].idxmax(), 'CF_Number']
        # 1 if same CF selected, 0 otherwise
        scheme_ranks[scheme_name].append(int(best_proposed_cfnum == best_other_cfnum))

print(f"\nSelection Agreement Rate with Proposed Scheme (per firm, n={len(firm_ids_list)}):")
for scheme_name, agreements in scheme_ranks.items():
    rate = np.mean(agreements) * 100
    print(f"  {scheme_name:30s}: {rate:.1f}% agreement")

# Also compute Spearman rank correlation of QS scores across all candidates
print(f"\nSpearman Rank Correlation of QualityScore with Proposed Scheme (all {len(metrics_df)} candidates):")
pool_C_metrics = metrics_df[~metrics_df['Immutable_Violated']].copy()
wv0, wp0, ws0, wr0, wrob0 = weight_schemes['Proposed (5,1,1,1,1)']
denom0 = wv0 + wp0 + ws0 + wr0 + wrob0
pool_C_metrics['QS_proposed'] = (
    wv0 * pool_C_metrics['Validity'] +
    wp0 * (1 - pool_C_metrics['Proximity']) +
    ws0 * pool_C_metrics['Sparsity'] +
    wr0 * pool_C_metrics['Realism'] +
    wrob0 * pool_C_metrics['Robustness']
) / denom0

for scheme_name, (wv, wp, ws, wr, wrob) in weight_schemes.items():
    if scheme_name == 'Proposed (5,1,1,1,1)':
        continue
    denom = wv + wp + ws + wr + wrob
    qs_other = (
        wv * pool_C_metrics['Validity'] +
        wp * (1 - pool_C_metrics['Proximity']) +
        ws * pool_C_metrics['Sparsity'] +
        wr * pool_C_metrics['Realism'] +
        wrob * pool_C_metrics['Robustness']
    ) / denom
    rho, pval = spearmanr(pool_C_metrics['QS_proposed'], qs_other)
    print(f"  {scheme_name:30s}: ρ = {rho:.4f}, p = {pval:.4f}")

# ============================================================================
# 9. Save All Results
# ============================================================================
df_A.to_csv('ablation_condA.csv', index=False)
df_B.to_csv('ablation_condB.csv', index=False)
df_C.to_csv('ablation_condC.csv', index=False)
metrics_df.to_csv('ablation_all_metrics.csv', index=False)

print("\n" + "=" * 80)
print("Ablation study complete. Files saved:")
print("  ablation_condA.csv / condB.csv / condC.csv / ablation_all_metrics.csv")
print("=" * 80)

Ablation Study: Selection Strategy Comparison (Same CF Pool)
  Condition A: No immutable constraint  + Random selection
  Condition B: Immutable constraint     + Random selection (first valid)
  Condition C: Immutable constraint     + Quality Score selection [Proposed]
Scaler & IsolationForest fitted on X_train (n=425).
Total CF candidates : 1060 rows
Unique firms        : 266
Threshold           : 0.6989

[Step 1] Computing metrics for all CF candidates...
Done. Total rows: 1060

[Step 2] Selecting best CF per firm under each condition...

ABLATION STUDY RESULTS (n=266 firms each condition)
                                          Validity                   Proximity         Sparsity          Realism       Robustness               QualityScore
A (No constraint + Random)         1.0000 ± 0.0000  518128.6631 ± 2499360.7413  0.4480 ± 0.0642  0.5915 ± 0.0010  1.0000 ± 0.0000  -57568.9582 ± 277706.7487
B (Immutable + Random)             1.0000 ± 0.0000  518128.6631 ± 2499360.7413  0.4480 

In [4]:
# ablation_condA.csv 로드 후 확인
df_A = pd.read_csv('ablation_condA.csv')

# Proximity 상위 이상치 확인
print(df_A.nlargest(5, 'Proximity')[['ID','CF_Number','Proximity','Sparsity','Realism']])

# Proximity 분포
print(f"\nProximity 중앙값: {df_A['Proximity'].median():.4f}")
print(f"Proximity 평균:   {df_A['Proximity'].mean():.4f}")
print(f"Proximity >100인 기업 수: {(df_A['Proximity']>100).sum()}")

           ID  CF_Number     Proximity  Sparsity   Realism
79    25114.0        1.0  1.252929e+07   0.43750  0.586756
228  120393.0        1.0  1.252929e+07   0.46875  0.586756
66   136849.0        1.0  1.252929e+07   0.71875  0.586756
117   24593.0        1.0  1.252929e+07   0.43750  0.586756
39    96148.0        1.0  1.252928e+07   0.43750  0.586756

Proximity 중앙값: 0.0219
Proximity 평균:   518128.6631
Proximity >100인 기업 수: 11


In [5]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

df_A = pd.read_csv('ablation_condA.csv')
df_B = pd.read_csv('ablation_condB.csv')
df_C = pd.read_csv('ablation_condC.csv')

METRICS = ['Proximity', 'Sparsity', 'Realism', 'QualityScore']

# ============================================================================
# 1. Summary Table: Median ± IQR
# ============================================================================
def median_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    return f"{series.median():.4f} [{q1:.4f}–{q3:.4f}]"

rows = {}
for label, df in [('A: No constraint + Random', df_A),
                  ('B: Immutable + Random',      df_B),
                  ('C: Immutable + QScore [Proposed]', df_C)]:
    rows[label] = {m: median_iqr(df[m]) for m in METRICS}

summary = pd.DataFrame(rows).T
print("=" * 80)
print("Ablation Study Results — Median [IQR] (n=266 firms per condition)")
print("=" * 80)
print(summary.to_string())

# ============================================================================
# 2. Wilcoxon Tests + Effect Size (A vs C, B vs C)
# ============================================================================
print("\nStatistical Tests (Wilcoxon signed-rank)")
print("-" * 60)

test_rows = {}
for m in METRICS:
    row = {}
    for label, (d1, d2) in [('A vs C', (df_A, df_C)), ('B vs C', (df_B, df_C))]:
        # Wilcoxon requires non-identical pairs
        diffs = d1[m].values - d2[m].values
        if np.all(diffs == 0):
            row[f'{label} p'] = 'identical'
            row[f'{label} r'] = '—'
        else:
            stat, p = wilcoxon(d1[m], d2[m])
            n = len(d1)
            r = 1 - (2 * stat) / (n * (n + 1))
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
            row[f'{label} p'] = f"{p:.4f} {sig}"
            row[f'{label} r'] = f"{r:.3f}"
    test_rows[m] = row

test_df = pd.DataFrame(test_rows).T
print(test_df.to_string())

# ============================================================================
# 3. Outlier 보고 (투명성 확보)
# ============================================================================
print("\nProximity Outlier Summary (Condition A — random selection)")
print("-" * 60)
for thresh in [100, 1000, 10000]:
    cnt = (df_A['Proximity'] > thresh).sum()
    print(f"  Proximity > {thresh:>6}: {cnt} firms ({cnt/len(df_A)*100:.1f}%)")

print(f"\n  Median Proximity (A): {df_A['Proximity'].median():.4f}")
print(f"  Median Proximity (C): {df_C['Proximity'].median():.4f}")
pct_improve = (1 - df_C['Proximity'].median() / df_A['Proximity'].median()) * 100
print(f"  Median improvement (C vs A): {pct_improve:.1f}%")

# ============================================================================
# 4. Sensitivity Analysis Summary
# ============================================================================
print("\nSensitivity Analysis — Weight Scheme Stability")
print("-" * 60)
sens_data = {
    'Weight Scheme': [
        'Proposed (Validity×5, others×1)',
        'Equal (all×1)',
        'Proximity-heavy (Validity×5, Proximity×3)',
        'Realism-heavy (Validity×5, Realism×3)',
        'Robustness-heavy (Validity×5, Robustness×3)',
    ],
    'Selection Agreement': ['—', '100.0%', '100.0%', '100.0%', '100.0%'],
    'Spearman ρ':          ['—', '1.0000', '0.9786', '1.0000', '1.0000'],
}
print(pd.DataFrame(sens_data).to_string(index=False))

Ablation Study Results — Median [IQR] (n=266 firms per condition)
                                               Proximity                Sparsity                 Realism            QualityScore
A: No constraint + Random         0.0219 [0.0095–0.0730]  0.4375 [0.4219–0.4531]  0.5917 [0.5917–0.5917]  0.8889 [0.8808–0.8930]
B: Immutable + Random             0.0219 [0.0095–0.0730]  0.4375 [0.4219–0.4531]  0.5917 [0.5917–0.5917]  0.8889 [0.8808–0.8930]
C: Immutable + QScore [Proposed]  0.0150 [0.0066–0.0354]  0.4375 [0.4219–0.4688]  0.5917 [0.5917–0.5917]  0.8905 [0.8862–0.8943]

Statistical Tests (Wilcoxon signed-rank)
------------------------------------------------------------
                A vs C p A vs C r    B vs C p B vs C r
Proximity     0.0000 ***    0.899  0.0000 ***    0.899
Sparsity      0.0000 ***    0.998  0.0000 ***    0.998
Realism       0.0009 ***    1.000  0.0009 ***    1.000
QualityScore  0.0000 ***    1.000  0.0000 ***    1.000

Proximity Outlier Summary (Condition A 